In [1]:
!mkdir '/content/dataset'

In [2]:
!apt-get install unzip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [3]:
!pip install pyyaml wandb roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 47.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


Download dataset yang ingin diproses

In [4]:
#!gdown 1ClaVQVepWcrw4dBesxMvoVwSdmL1THVW

In [5]:
#!gdown 1skrVBHdY0fhWKdi2zgGe1CyOmvx_7uQ2

Atau update dataset yang sudah ada di W&B
sebagai contoh:
wandb artifact get "FAIt (Food AI-based tracking)/[nama dataset]:latest" --root "/content/dataset/[nama dataset]"

In [4]:
# format --> wandb artifact get "FAIt (Food AI-based tracking)/<nama dataset>:latest" --root /content/dataset/<nama dataset>
#!wandb artifact get "FAIt (Food AI-based tracking)/New_Food_Dataset:latest" --root /content/dataset/New_Food_Dataset

Kalau ingin download dari roboflow, di halaman dataset ke tab dataset > download dataset > download dataset > show download code > jupyter

lihat section #dataset roboflow untuk detailnya

Unzip ke folder /dataset

In [ ]:
#!unzip -d "/content/dataset/Food Detection.v2i.yolo26" "/content/Food Detection.v2i.yolo26.zip"

In [ ]:
#!unzip -d "/content/dataset/Food_Detection_Project.v19i.yolo26" "Food_Detection_Project.v19i.yolo26.zip"

In [6]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import yaml
from collections import OrderedDict
import shutil
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime, timezone
from google.colab import userdata
import wandb

In [23]:
def get_classnames(dataset, dataset_dir='/content/dataset/'):
  with open(dataset_dir + dataset + "/data.yaml", 'r') as file:
    read = yaml.safe_load(file)
    classnames =  read['names']
    return classnames


In [7]:


def dataset_to_df(dataset_dir, classnames, image_dir="/images/", label_dir="/labels/", image_ext=".jpg"):
  rows = []
  le = LabelEncoder()
  le.fit(classnames)

  for train_set in os.listdir(dataset_dir):
    if os.path.isdir(dataset_dir +  train_set):
      for label_file in os.listdir(dataset_dir + train_set + label_dir):
          if not label_file.endswith(".txt"):
              continue

          file_name = os.path.splitext(label_file)[0]
          label_path = os.path.join(dataset_dir + train_set + label_dir, label_file)

          image_path = os.path.join(dataset_dir + train_set + image_dir, file_name + image_ext)

          image_exists = os.path.exists(image_path)

          with open(label_path, "r") as f:
              lines = f.readlines()

          for line in lines:
              line = line.strip()

              # Skip baris kosong
              if not line:
                  continue

              parts = line.split()

              # Minimal harus ada label + bounding box
              if len(parts) < 2:
                  rows.append({
                      "label_path": label_path,
                      "label_name": None,
                      "label": None,
                      "image_path": image_path,
                      "bounding_box": line,
                      "status": 0
                  })
                  continue

              label = int(parts[0])
              label_name = le.inverse_transform([label])[0]
              bounding_box = ' '.join(parts[1:])

              rows.append({
                  "label_path": label_path,
                  "label_name": label_name,
                  "label": label,
                  "image_path": image_path,
                  "bounding_box": bounding_box,
                  "status": 1 if image_exists else 0
              })

  # Buat DataFrame
  df = pd.DataFrame(rows)
  df['label_name'] = df['label_name'].str.lower()
  df['label_name'] = df['label_name'].str.replace('_', ' ')
  return df


In [8]:
def replace_dir(filename, dest):
  fname_split = filename.split('/')
  fname_split.pop(2)
  fname_split[2] = new_dataset_dir
  dest_fname = '/'.join(fname_split)
  dir_fname = '/'.join(fname_split[:-1])
  return dest_fname, dir_fname



In [9]:
def create_dataset_dir(dataset_dir, root='/content/'):
  for train_set in ['/train', '/test', '/valid']:
    for d_type in ['/images', '/labels']:
      new_dir = os.path.join(root + dataset_dir + train_set + d_type)
      if os.path.exists(new_dir) == False:
        os.makedirs(new_dir)


In [11]:
def get_dataset_list(dataset_dir='/content/dataset/'):
  dataset_list = []
  for dataset in os.listdir(dataset_dir):
    if os.path.isdir(dataset_dir + dataset):
      dataset_list.append(dataset)
  return dataset_list


In [10]:
def all_dataset_to_df(datasets_list, dataset_dir='/content/dataset/'):
  df_list = []
  for dataset in datasets_list:
    classnames = get_classnames(dataset )
    df = dataset_to_df(dataset_dir + dataset + "/", classnames)
    df_list.append(df)
  return df_list


In [12]:
def concat_df(df_list):
  combined_df = pd.concat(df_list, axis=0, join='outer', ignore_index=True)
  return combined_df

In [13]:
def get_new_labels(df):
  labeler = LabelEncoder().fit(df['label_name'])
  df['label'] = labeler.transform(df['label_name'])
  return labeler, df

In [14]:
class FlowStyleListDumper(yaml.Dumper):
    def represent_list(self, data):
        # Represent lists in flow style (inline)
        return self.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=True)

    def represent_ordereddict(self, data):
        return self.represent_mapping('tag:yaml.org,2002:map', data)

    def represent_str(self, data): # Added to force quoting of strings
        return self.represent_scalar('tag:yaml.org,2002:str', data, style='"')


def get_yaml_file(new_dataset_dir, labeler, datasets_list, nutri_fail=[]):

  data_yml = OrderedDict({
      "train" : "../train/images",
      "val" : "../valid/images",
      "test": "../test/images",
      "nc" : len(labeler.classes_),
      "names" : labeler.classes_.tolist(),
      "additional": {
          "datasets" : datasets_list,
          "datetime" : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
          "nutrition fail" : nutri_fail

      }
  })


  # Register the custom representers
  FlowStyleListDumper.add_representer(list, FlowStyleListDumper.represent_list)
  FlowStyleListDumper.add_representer(OrderedDict, FlowStyleListDumper.represent_ordereddict)
  FlowStyleListDumper.add_representer(str, FlowStyleListDumper.represent_str) # Register for string quoting

  with open("/content/" + new_dataset_dir + "/data.yaml", 'w') as file:
      yaml.dump(data_yml, file, Dumper=FlowStyleListDumper, default_flow_style=False, sort_keys=False, width=float('inf'))

In [15]:
def extract_angka_satuan(satuan):
    if satuan == '-':
        return '-'
    if '100 gram' in satuan.lower():
        return float(100)
    # Ambil nilai di dalam kurung, misal "1 porsi (250 g)" → "250 g"
    match_paren = re.search(r'\(([^)]+)\)', satuan)
    if match_paren:
        content_in_parentheses = match_paren.group(1) # e.g., "250 g"
        number_match = re.search(r'(\d+(?:[.,]\d+)?)', content_in_parentheses)
        if number_match:
            number_str = number_match.group(1).replace(',', '.')
            return float(number_str)

    general_number_match = re.search(r'(\d+(?:[.,]\d+)?)', satuan)
    if general_number_match:
        number_str = general_number_match.group(1).replace(',', '.')
        return float(number_str)

    return '-'


def scrape_nutrition_data(query_list, dataset_dir, root="/content/", base_url="https://www.fatsecret.co.id"):
    all_data = []
    base_url = "https://www.fatsecret.co.id"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for item in query_list:
        search_url = f"{base_url}/kalori-gizi/search?q={item}"

        row = {
            'nama makanan': item,
            'kalori': '-',
            'lemak': '-',
            'karbohidrat': '-',
            'protein': '-',
            'satuan': '-',
            'angka_satuan': '-',
            'link': '-',
            'status': '0'
        }

        try:
            # ======================
            # 1️⃣ Pencarian
            # ======================
            res = requests.get(search_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            # 🔹 Jika tidak ada hasil pencarian
            if soup.find('div', class_='searchNoResult'):
                print(f"❌ Tidak ditemukan: {item}")
                all_data.append(row)
                continue

            table = soup.find('table', class_='generic searchResult')
            if not table:
                print(f"❌ Tidak ditemukan (table kosong): {item}")
                all_data.append(row)
                continue

            first_link = table.find('a', class_='prominent')
            if not first_link:
                print(f"❌ Tidak ditemukan (link kosong): {item}")
                all_data.append(row)
                continue

            detail_url = base_url + first_link['href']

            # ======================
            # 2️⃣ Masuk halaman detail
            # ======================
            res = requests.get(detail_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            # ======================
            # 3️⃣ Cek link 100 gram
            # ======================
            link_100g = soup.find('a', string=lambda s: s and "100 gram" in s.lower())

            if link_100g and link_100g.get('href'):
                detail_url = base_url + link_100g['href']
                res = requests.get(detail_url, headers=headers)
                soup = BeautifulSoup(res.text, 'html.parser')

            row['link'] = detail_url

            # ======================
            # 4️⃣ Ambil satuan dari div
            # ======================
            serving_div = soup.select_one('div.serving_size.black.serving_size_value')

            if serving_div:
                row['satuan'] = serving_div.text.strip()
            else:
                row['satuan'] = '-'

            # ✅ Isi angka_satuan berdasarkan satuan
            row['angka_satuan'] = extract_angka_satuan(row['satuan'])

            # ======================
            # 5️⃣ Ambil nilai gizi
            # ======================
            facts = soup.find_all('td', class_='fact')

            for fact in facts:
                title_el = fact.find('div', class_='factTitle')
                value_el = fact.find('div', class_='factValue')

                if title_el and value_el:
                    title = title_el.text.strip().lower()
                    value = value_el.text.strip()

                    if 'kal' in title:
                        row['kalori'] = extract_angka_satuan(value)
                    elif 'lemak' in title:
                        row['lemak'] = extract_angka_satuan(value)
                    elif 'karb' in title:
                        row['karbohidrat'] = extract_angka_satuan(value)
                    elif 'prot' in title:
                        row['protein'] = extract_angka_satuan(value)

            if facts:
                row['status'] = '1'
            else:
                print(f"⚠️ Data gizi kosong: {item}")

        except Exception as e:
            print(f"Error pada {item}: {e}")
            row['status'] = 'Error'

        all_data.append(row)
        time.sleep(1.5)

    df = pd.DataFrame(all_data)
    df.to_csv(root + dataset_dir + '/hasil_gizi_100gram.csv', index=False, encoding='utf-8-sig')
    nutri_success, nutri_fail = df[df['status'] == '1'], df[df['status'] == '0']
    return nutri_success, nutri_fail

    print("\nSelesai! Data berhasil disimpan dengan kolom satuan.")


def preprocess_query(query_list):
    cleaned_list = []
    for item in query_list:
        item = item.replace("_", " ")
        item = item.strip()
        cleaned_list.append(item)
    return cleaned_list


In [16]:
def df_to_dataset(df, new_dataset_dir, labeler, datasets_list):
  create_dataset_dir(new_dataset_dir)
  for filenames in df['label_path'].unique():
    curr_df = df.loc[df['label_path'] == filenames]
    image_path = curr_df['image_path'].values[0]

    dest_label, dir_label = replace_dir(filenames, new_dataset_dir)
    curr_df.to_csv(dest_label, sep=' ', columns=['label', 'bounding_box'], header=False, index=False)
    with open(dest_label, 'r+') as f:
        data = f.read().rstrip('\n')
        data = data.replace('"', "")
        f.seek(0)
        f.write(data)
        f.truncate()
    dest_image, dir_image = replace_dir(image_path, new_dataset_dir)
    if os.path.exists(dest_image) == False:
      shutil.copy2(image_path, dest_image)
  daftar_cari = preprocess_query(labeler.classes_.tolist())
  nutri_success, nutri_fail = scrape_nutrition_data(daftar_cari, new_dataset_dir)
  get_yaml_file(new_dataset_dir, labeler, datasets_list, nutri_fail['nama makanan'].tolist())

  return nutri_success, nutri_fail







In [17]:
class WandbHandler:
  def __init__(self, project_name, root='/content/'):
    self.project_name = project_name
    self.root = root
    wandb.login(key=userdata.get('WANDB_API_KEY'))

  def download_from_wandb(self, dataset_name):
    with wandb.init(project=self.project_name, job_type='download_dataset') as run:
      artifact = run.use_artifact(f"{self.project_name}/{dataset_name}:latest")
      dataset_root = f"{self.root}dataset/{dataset_name}"
      artifact.download(root=dataset_root)
    return dataset_root


  def upload_to_wandb(self, dataset_dir):
    artifact = wandb.Artifact(name=dataset_dir, type="dataset")
    artifact.add_dir(self.root + dataset_dir)
    with wandb.init(project=self.project_name, job_type='upload_dataset') as run:
      run.log_artifact(artifact)

  def update_dataset(self, dataset_dir):
    with wandb.init(project=self.project_name, job_type='update_dataset') as run:
      artifact = wandb.Artifact(name=dataset_dir, type="dataset")
      artifact.add_dir(self.root + dataset_dir)
      run.log_artifact(artifact)


  def update_file(self, dataset_dir, file_name):
    with wandb.init(project=self.project_name, job_type='update_file') as run:
      saved_artifact = run.use_artifact(dataset_dir+":latest")
      draft_artifact = saved_artifact.new_draft()

      draft_artifact.remove(saved_artifact.get_entry(file_name))
      draft_artifact.add_file(local_path= self.root + dataset_dir + "/" + file_name, name=file_name)

      draft_artifact.save()





# Dataset Roboflow
Kalau ingin download dari roboflow, di halaman dataset ke tab dataset > download dataset > download dataset > show download code > jupyter

Copy paste codenya, tambahkan argument location pada version.download() dan isi dengan "/content/dataset/[Nama Dataset]"



Nama dataset ada di .project(), sebgai contoh:


dataset = version.download("yolo26", location="/content/dataset/Food Computer Vision Dataset")

In [18]:



from roboflow import Roboflow
rf = Roboflow(api_key="myPNaXQCDLYlKH3dlyVL")
project = rf.workspace("universitas-sumatera-utara-h9u4i").project("food-0yybl")
version = project.version(2)
dataset = version.download("yolo26", location="/content/dataset/Food Computer Vision Dataset")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/dataset/Food Computer Vision Dataset in yolo26:: 100%|██████████| 14956/14956 [00:03<00:00, 4355.42it/s]


Nama dataset baru, tolong sesuaikan

In [19]:
new_dataset_dir = 'Food_Computer_Vision_Dataset'

In [20]:
wandb_handler = WandbHandler('FAIt (Food AI-based tracking)')

invalid escape sequence '\/'
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: b10g004dicoding (b10g004dicoding-dicoding-batch-10-g004) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [21]:
datasets_list = get_dataset_list()

In [24]:
dataset_df_list = all_dataset_to_df(datasets_list)

In [25]:
combined_df = concat_df(dataset_df_list)

In [26]:
dataset_df_list[0]

,label_path,label_name,label,image_path,bounding_box,status
0,/content/dataset/Food Computer Vision Dataset/...,bihun,8,/content/dataset/Food Computer Vision Dataset/...,0.46875 0.5033333333333333 0.595 0.63333333333...,1
1,/content/dataset/Food Computer Vision Dataset/...,rendang,31,/content/dataset/Food Computer Vision Dataset/...,0.5602115384615385 0.39173076923076927 0.81594...,1
2,/content/dataset/Food Computer Vision Dataset/...,pempek,29,/content/dataset/Food Computer Vision Dataset/...,0.49921875 0.44140625 0.7125 0.3859375,1
3,/content/dataset/Food Computer Vision Dataset/...,pempek,29,/content/dataset/Food Computer Vision Dataset/...,0.3046875 0.81328125 0.3234375 0.2625,1
4,/content/dataset/Food Computer Vision Dataset/...,pempek,29,/content/dataset/Food Computer Vision Dataset/...,0.7046875 0.8109375 0.3203125 0.2625,1
...,...,...,...,...,...,...
13458,/content/dataset/Food Computer Vision Dataset/...,tahu,41,/content/dataset/Food Computer Vision Dataset/...,0.6201923076923077 0.7584134615384616 0.25 0.3...,1
13459,/content/dataset/Food Computer Vision Dataset/...,tahu,41,/content/dataset/Food Computer Vision Dataset/...,0.4735576923076923 0.8533653846153846 0.307692...,1
13460,/content/dataset/Food Computer Vision Dataset/...,tahu,41,/content/dataset/Food Computer Vision Dataset/...,0.22115384615384615 0.29447115384615385 0.2644...,1
13461,/content/dataset/Food Computer Vision Dataset/...,telur,42,/content/dataset/Food Computer Vision Dataset/...,0.3542609853528629 0.2991917293233083 0.361731...,1


In [ ]:
dataset_df_list[1]

In [27]:
new_label, combined_df = get_new_labels(combined_df)

In [31]:
nutri_success, nutri_fail = df_to_dataset(combined_df, new_dataset_dir, new_label, datasets_list)

⚠️ Data gizi kosong: nasi
⚠️ Data gizi kosong: nasi putih
⚠️ Data gizi kosong: tumis kangkung


In [ ]:
nutri_success

In [32]:
nutri_fail

,nama makanan,kalori,lemak,karbohidrat,protein,satuan,angka_satuan,link,status
24,nasi,-,-,-,-,-,-,https://www.fatsecret.co.id/kalori-gizi/umum/n...,0
26,nasi putih,-,-,-,-,-,-,https://www.fatsecret.co.id/kalori-gizi/umum/n...,0
46,tumis kangkung,-,-,-,-,-,-,https://www.fatsecret.co.id/kalori-gizi/umum/t...,0


In [35]:
scrape_success, scrape_fail = scrape_nutrition_data(preprocess_query(nutri_fail['nama makanan'].values.tolist()), new_dataset_dir)

In [36]:
scrape_success

,nama makanan,kalori,lemak,karbohidrat,protein,satuan,angka_satuan,link,status
0,nasi,129.0,0.28,27.90,2.66,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/n...,1
1,nasi putih,129.0,0.28,27.90,2.66,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/n...,1
2,tumis kangkung,98.0,8.71,3.99,2.56,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/t...,1


In [ ]:
# wandb_handler.upload_to_wandb(new_dataset_dir)

In [ ]:
# update = {
#     13 : {
#     "nama makanan": "cucumber",
#     "kalori": 12,
#     "lemak": "0,16g",
#     "karbohidrat": "2.16g",
#     "protein": "0.59g",
#     "satuan": "100 gram (g)",
#     "link": "https://foods.fatsecret.com/calories-nutrition/usda/cucumber-(peeled)?portionid=59108&portionamount=100.000",
#     "status": 1
#     }
# }

In [ ]:
# for idx, values in update.items():
#     nutri_fail.loc[idx] = values

In [ ]:
nutri_fail

,nama makanan,kalori,lemak,karbohidrat,protein,satuan,link,status
13,cucumber,12,"0,16g",2.16g,0.59g,100 gram (g),https://foods.fatsecret.com/calories-nutrition...,1


In [37]:
new_nutri = concat_df([nutri_success, scrape_success])

In [38]:
new_nutri

,nama makanan,kalori,lemak,karbohidrat,protein,satuan,angka_satuan,link,status
0,alpukat,160.0,14.66,8.53,2.0,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/a...,1
1,ayam,260.0,14.55,10.76,21.93,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/a...,1
2,ayam goreng,260.0,14.55,10.76,21.93,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/a...,1
3,ayam panggang,167.0,6.63,0.0,25.01,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/d...,1
4,bakso ayam,161.0,5.62,6.94,19.39,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/b...,1
5,bakwan,228.0,19.31,11.23,3.31,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/b...,1
6,batagor,290.0,14.92,29.14,10.28,100 g,100.0,https://www.fatsecret.co.id/kalori-gizi/batago...,1
7,bayam,40.0,2.25,3.68,2.88,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/b...,1
8,bihun,109.0,0.2,24.9,0.91,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/b...,1
9,brokoli,54.0,2.69,6.98,2.3,100 gram (g),100.0,https://www.fatsecret.co.id/kalori-gizi/umum/b...,1


In [39]:
new_nutri.to_csv('/content/' + new_dataset_dir + '/hasil_gizi_100gram.csv', index=False, encoding='utf-8-sig')

In [40]:
get_yaml_file(new_dataset_dir, new_label, datasets_list)

In [41]:
wandb_handler.upload_to_wandb(new_dataset_dir)

wandb: Adding directory to artifact (/content/Food_Computer_Vision_Dataset)... Done. 13.7s


In [ ]:
wandb_handler.update_dataset(new_dataset_dir)

In [ ]:
wandb_handler.update_file(new_dataset_dir,"data.yaml")